# Join Tables

This notebook connects cleaned datasets to CBS neighbourhoods.

Inputs:
- DS1: bike parking facilities
- DS2: cycling accidents
- DS3: CBS neighbourhoods
- DS4: OSM bike lanes

Main idea:
- DS3 neighbourhoods are used as the spatial hub.
- Parking facilities are assigned to neighbourhoods.
- Accidents are assigned to neighbourhoods.
- Bike lane length is aggregated per neighbourhood.

Outputs:
- parking_neighbourhood_join.csv
- parking_neighbourhood_summary.csv
- accident_neighbourhood_join.csv
- accident_neighbourhood_summary.csv
- lane_neighbourhood_summary.csv
- neighbourhood_summary.csv

# Import packages


In [1]:
import pandas as pd
import geopandas as gpd
from shapely import wkt

# Load datasets

In [3]:
ds1 = pd.read_csv("../data/clean/ds1.csv")
ds2 = pd.read_csv("../data/clean/ds2.csv")
ds3 = pd.read_csv("../data/clean/ds3.csv")

lanes = gpd.read_file("../data/clean/eindhoven_bike_lanes_basic_clean.gpkg")

print("DS1 parking:", ds1.shape)
print("DS2 accidents:", ds2.shape)
print("DS3 neighbourhoods:", ds3.shape)
print("DS4 lanes:", lanes.shape)

DS1 parking: (7, 8)
DS2 accidents: (1653, 40)
DS3 neighbourhoods: (111, 40)
DS4 lanes: (4722, 169)


# Prepare neighbourhood polygons

DS3 is the central table in this workflow. Because it is stored as a CSV file, the geometry column is currently represented as text strings rather than actual polygon objects. Therefore, we need to convert these strings back into real polygon geometries before doing spatial joins.

In [4]:
ds3["geometry"] = ds3["geometry"].apply(
    lambda x: wkt.loads(x) if isinstance(x, str) else x
)

neighbourhoods = gpd.GeoDataFrame(
    ds3,
    geometry="geometry",
    crs="EPSG:28992"
)

neighbourhoods["BUURTCODE"] = neighbourhoods["BUURTCODE"].astype(int).astype(str)

print(neighbourhoods.crs)
print(neighbourhoods.geometry.geom_type.value_counts())
neighbourhoods.head()

EPSG:28992
Polygon    111
Name: count, dtype: int64


,BUURTNAAM,BUURTCODE,WIJKNAAM,STADSDEELNAAM,geo_shape,Inwoners|2005,Inwoners|2006,Inwoners|2007,Inwoners|2008,Inwoners|2009,...,pop_growth_rate,pop_growth_2005_2025,area_km2,population,OAD,pop_density,urban_pressure,growth_pressure,pop_density_z,OAD_z
0,Achtse Barrier-Gunterslaer,522,Wijk Achtse Molen,Stadsdeel Woensel-Noord,"{""coordinates"": [[[5.45625536423871, 51.479033...",4092.0,4029.0,3954.0,3932.0,3901.0,...,-0.077957,-319.0,1.015387,3773.0,3715.825734,3715.825734,30603.134701,-289.674587,-0.296210,-0.296210
1,Achtse Barrier-Hoeven,524,Wijk Achtse Molen,Stadsdeel Woensel-Noord,"{""coordinates"": [[[5.444884639883207, 51.48575...",4467.0,4425.0,4316.0,4341.0,4304.0,...,-0.130065,-581.0,0.744209,3886.0,5221.648554,5221.648554,43158.977043,-679.153304,0.171233,0.171233
2,Achtse Barrier-Spaaihoef,523,Wijk Achtse Molen,Stadsdeel Woensel-Noord,"{""coordinates"": [[[5.439586926395638, 51.48854...",4876.0,4900.0,4855.0,4874.0,4818.0,...,-0.072190,-352.0,1.015942,4524.0,4453.010370,4453.010370,37482.648615,-321.464243,-0.067370,-0.067370
3,Barrier,423,Wijk Erp,Stadsdeel Woensel-Zuid,"{""coordinates"": [[[5.454851547237404, 51.46224...",1601.0,1639.0,1738.0,1948.0,2154.0,...,0.274204,439.0,0.257474,2040.0,7923.122667,7923.122667,60383.664147,2172.548939,1.009834,1.009834
4,BeA2,631,Wijk Meerhoven,Stadsdeel Strijp,"{""coordinates"": [[[5.393934485995225, 51.46433...",144.0,61.0,94.0,52.0,40.0,...,-0.840278,-121.0,2.620186,23.0,8.778001,8.778001,27.896961,-7.375959,-1.446965,-1.446965


# Prepare parking points 

DS1 contains longitude and latitude columns. Therefore, we can convert each parking location into a point geometry.

In [14]:
parking = gpd.GeoDataFrame(
    ds1,
    geometry=gpd.points_from_xy(ds1["lon"], ds1["lat"]),
    crs="EPSG:4326"
)

parking = parking.to_crs("EPSG:28992")

print(parking.crs)
print(parking.geometry.geom_type.value_counts())
parking.head()

EPSG:28992
Point    7
Name: count, dtype: int64


,facility_id,name,address,type,lon,lat,geo_point,capacity,geometry
0,2,Fietsenstalling Winkelcentrum WoensXL,"Winkelcentrum Woensel 500-510, 5625 AA Eindhoven",Gratis fietsenstalling met toezicht,5.473288,51.469785,"51.46978533335247, 5.4732884460401",200.0,POINT (160981.588 386750.698)
1,7,"Pop-up fietsenstalling Mercado, tijdelijke sta...","Smalle Haven 109, 5611 EH Eindhoven",Gratis fietsenstalling met toezicht,5.481647,51.436996,"51.43699601116056, 5.481646994194696",NaN,POINT (161567.096 383103.57)
2,5,Fietsenstalling NS-station noordzijde,"Neckerspoel, 5611 AD Eindhoven",Bewaakte NS stalling (tegen betaling),5.478307,51.443333,"51.44333306558167, 5.478306836629171",200.0,POINT (161333.963 383808.277)
3,1,Fietsenstalling Heuvel,"Ten Hagestraat 14, 5611 EJ Eindhoven",Gratis fietsenstalling met toezicht,5.480458,51.437850,"51.437850419905324, 5.480458223533857",900.0,POINT (161484.315 383198.517)
4,3,Fietsenstalling 18 Septemberplein,"18 Septemberplein, 5611 AK, Eindhoven",Gratis fietsenstalling met toezicht,5.477071,51.441229,"51.44122887025675, 5.477070707060177",1400.0,POINT (161248.312 383574.077)


# Prepare accident points


Similarly, DS2 also contains longitude and latitude columns. Therefore, we can convert each accident record into a point geometry.

In [15]:
from shapely import wkt
import geopandas as gpd

# Standardize accident outcome categories
outcome_mapping = {
    "Uitsluitend materiele schade": "property_damage_only",
    "Uitsluitend materiële schade": "property_damage_only",
    "Letsel": "injury",
    "Dodelijk": "fatal",
    "property_damage_only": "property_damage_only",
    "injury": "injury",
    "fatal": "fatal"
}

ds2["accident_outcome_standardized"] = (
    ds2["accident_outcome"]
    .map(outcome_mapping)
    .fillna(ds2["accident_outcome"])
)

print(ds2["accident_outcome_standardized"].value_counts(dropna=False))

# Convert geometry from WKT to shapely geometry if needed
ds2["geometry"] = ds2["geometry"].apply(
    lambda x: wkt.loads(x) if isinstance(x, str) else x
)

# Create GeoDataFrame in Dutch projected coordinate system
accidents = gpd.GeoDataFrame(
    ds2,
    geometry="geometry",
    crs="EPSG:28992"
)

# Extract longitude and latitude for export / checking
accidents_wgs84 = accidents.to_crs("EPSG:4326")
accidents["longitude"] = accidents_wgs84.geometry.x
accidents["latitude"] = accidents_wgs84.geometry.y

print(accidents.crs)
print(accidents.geometry.geom_type.value_counts())

accidents[
    [
        "accident_id",
        "accident_year",
        "accident_outcome_standardized",
        "longitude",
        "latitude",
        "geometry"
    ]
].head()

accident_outcome_standardized
property_damage_only    1301
injury                   348
fatal                      4
Name: count, dtype: int64
EPSG:28992
Point    1653
Name: count, dtype: int64


,accident_id,accident_year,accident_outcome_standardized,longitude,latitude,geometry
0,20220067354,2022,property_damage_only,5.396512,51.436566,POINT (155647.446 383051.535)
1,20230008779,2023,property_damage_only,5.472431,51.434861,POINT (160926.535 382865.3)
2,20230007858,2023,property_damage_only,5.475478,51.441108,POINT (161137.622 383560.536)
3,20230009169,2023,property_damage_only,5.514564,51.441977,POINT (163854.947 383661.251)
4,20240034437,2024,property_damage_only,5.481209,51.444889,POINT (161535.472 383981.587)


In [13]:
required_ds2_cols = [
    "accident_id",
    "accident_year",
    "accident_outcome_standardized",
    "longitude",
    "latitude"
]

missing_ds2_cols = [col for col in required_ds2_cols if col not in accidents.columns]

print("Missing DS2 columns:", missing_ds2_cols)

Missing DS2 columns: []


# Prepare bike lanes

DS4 was originally stored in WGS84, which uses longitude and latitude coordinates. Since length calculations should not be done directly on degree-based coordinates, we convert DS4 to the Dutch projected coordinate system so that distances are measured in metres.

In [16]:
lanes = lanes.to_crs("EPSG:28992")

print(lanes.crs)
print(lanes.geometry.geom_type.value_counts())
lanes.head()

EPSG:28992
LineString    4722
Name: count, dtype: int64


,element,id,button_operated,crossing,crossing:island,crossing:markings,crossing:signals,highway,tactile_paving,traffic_signals:sound,...,parking:lane:both,check_date:cycleway,sidewalk:left:surface,old_name:1974-12-31--2023-12-06,maxaxleload,priority,placement:end,placement:start,throughroute:bicycle,geometry
0,way,4260673,NaN,NaN,NaN,NaN,NaN,cycleway,NaN,NaN,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LINESTRING (161892.606 384083.97, 161893.284 3..."
1,way,4260684,NaN,NaN,NaN,NaN,NaN,cycleway,NaN,NaN,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LINESTRING (162853.715 384171.535, 162851.738 ..."
2,way,4260685,NaN,NaN,NaN,NaN,NaN,cycleway,NaN,NaN,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LINESTRING (162872.086 384262.4, 162880.863 38..."
3,way,4260687,NaN,NaN,NaN,NaN,NaN,cycleway,NaN,NaN,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LINESTRING (164602.273 385120.588, 164600.776 ..."
4,way,4260692,NaN,NaN,NaN,NaN,NaN,cycleway,NaN,NaN,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LINESTRING (164787.969 385505.974, 164790.643 ..."


# Join parking to neighbourhoods

In [17]:
parking_join = gpd.sjoin(
    parking,
    neighbourhoods[["BUURTCODE", "BUURTNAAM", "geometry"]],
    how="left",
    predicate="within"
)

parking_join[
    ["facility_id", "name", "capacity", "BUURTCODE", "BUURTNAAM"]
]

,facility_id,name,capacity,BUURTCODE,BUURTNAAM
0,2,Fietsenstalling Winkelcentrum WoensXL,200.0,515,Winkelcentrum
1,7,"Pop-up fietsenstalling Mercado, tijdelijke sta...",NaN,111,Binnenstad
2,5,Fietsenstalling NS-station noordzijde,200.0,114,Fellenoord
3,1,Fietsenstalling Heuvel,900.0,111,Binnenstad
4,3,Fietsenstalling 18 Septemberplein,1400.0,111,Binnenstad
5,4,Fietsenstalling NS-station zuidzijde,5000.0,111,Binnenstad
6,6,"Pop-up fietsenstalling Kerkstraat, tijdelijke ...",NaN,111,Binnenstad


Check whether there are any parking locations that were not matched:

In [15]:
unmatched_parking = parking_join[parking_join["BUURTCODE"].isna()]

print("Unmatched parking facilities:", len(unmatched_parking))
unmatched_parking[["facility_id", "name", "lon", "lat"]]

Unmatched parking facilities: 0


,facility_id,name,lon,lat


# Save parking join table

In [18]:
parking_join_table = parking_join[
    [
        "facility_id",
        "name",
        "address",
        "type",
        "capacity",
        "lon",
        "lat",
        "BUURTCODE",
        "BUURTNAAM"
    ]
].copy()

parking_join_table.to_csv(
    "../data/clean/parking_neighbourhood_join.csv",
    index=False
)

parking_join_table

,facility_id,name,address,type,capacity,lon,lat,BUURTCODE,BUURTNAAM
0,2,Fietsenstalling Winkelcentrum WoensXL,"Winkelcentrum Woensel 500-510, 5625 AA Eindhoven",Gratis fietsenstalling met toezicht,200.0,5.473288,51.469785,515,Winkelcentrum
1,7,"Pop-up fietsenstalling Mercado, tijdelijke sta...","Smalle Haven 109, 5611 EH Eindhoven",Gratis fietsenstalling met toezicht,NaN,5.481647,51.436996,111,Binnenstad
2,5,Fietsenstalling NS-station noordzijde,"Neckerspoel, 5611 AD Eindhoven",Bewaakte NS stalling (tegen betaling),200.0,5.478307,51.443333,114,Fellenoord
3,1,Fietsenstalling Heuvel,"Ten Hagestraat 14, 5611 EJ Eindhoven",Gratis fietsenstalling met toezicht,900.0,5.480458,51.437850,111,Binnenstad
4,3,Fietsenstalling 18 Septemberplein,"18 Septemberplein, 5611 AK, Eindhoven",Gratis fietsenstalling met toezicht,1400.0,5.477071,51.441229,111,Binnenstad
5,4,Fietsenstalling NS-station zuidzijde,"Stationsplein 22, 5611 AC Eindhoven",Bewaakte NS stalling (tegen betaling),5000.0,5.480394,51.442427,111,Binnenstad
6,6,"Pop-up fietsenstalling Kerkstraat, tijdelijke ...","Kerkstraat, 5611 GH Eindhoven",Gratis fietsenstalling met toezicht,NaN,5.478883,51.437059,111,Binnenstad


This table answers the question: which neighbourhood does each bike parking facility belong to?

# Aggregate parking by neighbourhood

In [19]:
parking_summary = (
    parking_join
    .groupby(["BUURTCODE", "BUURTNAAM"], as_index=False)
    .agg(
        parking_count=("facility_id", "count"),
        parking_capacity=("capacity", "sum")
    )
)

parking_summary.to_csv(
    "../data/clean/parking_neighbourhood_summary.csv",
    index=False
)

parking_summary.sort_values("parking_capacity", ascending=False)

,BUURTCODE,BUURTNAAM,parking_count,parking_capacity
0,111,Binnenstad,5,7300.0
1,114,Fellenoord,1,200.0
2,515,Winkelcentrum,1,200.0


This table answers the question: how many parking facilities each neighbourhood has and what the total parking capacity is.

# Join accidents to neighbourhoods

In [22]:
accident_join = gpd.sjoin(
    accidents,
    neighbourhoods[["BUURTCODE", "BUURTNAAM", "geometry"]],
    how="left",
    predicate="within"
)

accident_join[
    [
        "accident_id",
        "accident_year",
        "accident_outcome_standardized",
        "street_name",
        "BUURTCODE",
        "BUURTNAAM"
    ]
].head()

,accident_id,accident_year,accident_outcome_standardized,street_name,BUURTCODE,BUURTNAAM
0,20220067354,2022,property_damage_only,Waterlinie,635,Waterrijk
1,20230008779,2023,property_damage_only,Mauritsstraat,112,Bergen
2,20230007858,2023,property_damage_only,Mathildelaan,113,Witte Dame
3,20230009169,2023,property_damage_only,Generaal Bothastraat,322,Doornakkers-Oost
4,20240034437,2024,property_damage_only,Professor Dr Dorgelolaan,114,Fellenoord


Check whether there are any accident records that were not matched:

In [23]:
unmatched_accidents = accident_join[accident_join["BUURTCODE"].isna()]

print("Unmatched accidents:", len(unmatched_accidents))
print("Total accidents:", len(accident_join))
print(f"Unmatched rate: {len(unmatched_accidents) / len(accident_join) * 100:.2f}%")

unmatched_accidents[
    ["accident_id", "longitude", "latitude", "street_name"]
].head()

Unmatched accidents: 32
Total accidents: 1653
Unmatched rate: 1.94%


,accident_id,longitude,latitude,street_name
14,20240043980,5.423373,51.463189,Oirschotsedijk
46,20240073425,5.422175,51.462465,Oirschotsedijk
87,20220089829,5.421384,51.465067,Anthony Fokkerweg
104,20240034361,5.462045,51.406810,High Tech Campus
119,20240136331,5.418531,51.463915,Anthony Fokkerweg


Out of 1653 accident records, 32 could not be matched to a CBS neighbourhood using the `within` predicate, accounting for 1.94% of the records. These records are excluded from the neighbourhood-level aggregation because their locations may fall outside the CBS neighbourhood polygons or in boundary areas, making it difficult to assign them reliably to a specific neighbourhood.

# Save accident join table

In [24]:
desired_accident_join_cols = [
    "accident_id",
    "accident_year",
    "accident_outcome_standardized",
    "accident_nature",
    "road_situation",
    "speed_limit",
    "street_name",
    "longitude",
    "latitude",
    "BUURTCODE",
    "BUURTNAAM"
]

existing_accident_join_cols = [
    col for col in desired_accident_join_cols
    if col in accident_join.columns
]

accident_join_table = accident_join[existing_accident_join_cols].copy()

accident_join_table.to_csv(
    "../data/clean/accident_neighbourhood_join.csv",
    index=False
)

accident_join_table.head()

,accident_id,accident_year,accident_outcome_standardized,accident_nature,road_situation,speed_limit,street_name,longitude,latitude,BUURTCODE,BUURTNAAM
0,20220067354,2022,property_damage_only,Onbekend,Rechte weg,30.0,Waterlinie,5.396512,51.436566,635,Waterrijk
1,20230008779,2023,property_damage_only,Flank,Rechte weg,50.0,Mauritsstraat,5.472431,51.434861,112,Bergen
2,20230007858,2023,property_damage_only,Flank,"Kruispunt, 3 takken",50.0,Mathildelaan,5.475478,51.441108,113,Witte Dame
3,20230009169,2023,property_damage_only,Onbekend,Rechte weg,30.0,Generaal Bothastraat,5.514564,51.441977,322,Doornakkers-Oost
4,20240034437,2024,property_damage_only,Flank,"Kruispunt, 4 takken",30.0,Professor Dr Dorgelolaan,5.481209,51.444889,114,Fellenoord


This table answers the question: in which neighbourhood did each accident occur?

# Aggregate accidents by neighbourhood

In [26]:
matched_accident_join = accident_join.dropna(subset=["BUURTCODE"]).copy()

print(matched_accident_join["accident_outcome_standardized"].value_counts(dropna=False))

accident_summary = (
    matched_accident_join
    .groupby(["BUURTCODE", "BUURTNAAM"], as_index=False)
    .agg(
        accident_count=("accident_id", "count"),
        injury_count=(
            "accident_outcome_standardized",
            lambda x: (x == "injury").sum()
        ),
        fatal_count=(
            "accident_outcome_standardized",
            lambda x: (x == "fatal").sum()
        ),
        property_damage_only_count=(
            "accident_outcome_standardized",
            lambda x: (x == "property_damage_only").sum()
        )
    )
)

accident_summary.to_csv(
    "../data/clean/accident_neighbourhood_summary.csv",
    index=False
)

accident_summary.sort_values("accident_count", ascending=False).head(10)

accident_outcome_standardized
property_damage_only    1276
injury                   341
fatal                      4
Name: count, dtype: int64


,BUURTCODE,BUURTNAAM,accident_count,injury_count,fatal_count,property_damage_only_count
0,111,Binnenstad,89,18,0,71
82,621,Hurk,39,7,1,31
30,321,Doornakkers-West,38,7,1,30
5,211,Irisbuurt,38,7,0,31
47,425,Rapenland,38,4,0,34
40,412,Hemelrijken,37,10,0,27
107,731,Genderbeemd,37,2,0,35
41,413,Gildebuurt,37,5,0,32
63,522,Achtse Barrier-Gunterslaer,36,9,0,27
3,114,Fellenoord,35,7,0,28


In [27]:
accident_join_table = accident_join[
    [
        "accident_id",
        "accident_year",
        "accident_outcome_standardized",
        "accident_nature",
        "road_situation",
        "speed_limit",
        "street_name",
        "longitude",
        "latitude",
        "BUURTCODE",
        "BUURTNAAM"
    ]
].copy()

accident_join_table.to_csv(
    "../data/clean/accident_neighbourhood_join.csv",
    index=False
)

accident_join_table.head()

,accident_id,accident_year,accident_outcome_standardized,accident_nature,road_situation,speed_limit,street_name,longitude,latitude,BUURTCODE,BUURTNAAM
0,20220067354,2022,property_damage_only,Onbekend,Rechte weg,30.0,Waterlinie,5.396512,51.436566,635,Waterrijk
1,20230008779,2023,property_damage_only,Flank,Rechte weg,50.0,Mauritsstraat,5.472431,51.434861,112,Bergen
2,20230007858,2023,property_damage_only,Flank,"Kruispunt, 3 takken",50.0,Mathildelaan,5.475478,51.441108,113,Witte Dame
3,20230009169,2023,property_damage_only,Onbekend,Rechte weg,30.0,Generaal Bothastraat,5.514564,51.441977,322,Doornakkers-Oost
4,20240034437,2024,property_damage_only,Flank,"Kruispunt, 4 takken",30.0,Professor Dr Dorgelolaan,5.481209,51.444889,114,Fellenoord


# Intersect bike lanes with neighbourhoods

A single bike lane may pass through multiple neighbourhoods. Therefore, we split the bike lane geometries by neighbourhood boundaries and calculate the length of each segment separately.

In [28]:
lane_intersections = gpd.overlay(
    lanes,
    neighbourhoods[["BUURTCODE", "BUURTNAAM", "geometry"]],
    how="intersection"
)

lane_intersections["lane_length_m"] = lane_intersections.geometry.length

print("Lane intersections:", lane_intersections.shape)
lane_intersections[["id", "highway", "BUURTCODE", "BUURTNAAM", "lane_length_m"]].head()

Lane intersections: (5467, 172)


,id,highway,BUURTCODE,BUURTNAAM,lane_length_m
0,4260673,cycleway,115,TU-terrein,36.485973
1,4260684,cycleway,115,TU-terrein,7.135659
2,4260685,cycleway,336,Karpen,20.118614
3,4260685,cycleway,115,TU-terrein,10.100727
4,4260687,cycleway,337,Koudenhoven,149.911607


In [29]:
desired_lane_join_cols = [
    "id",
    "element",
    "highway",
    "bicycle",
    "cycleway",
    "cycleway:left",
    "cycleway:right",
    "cycleway:both",
    "source_query",
    "included_reason",
    "BUURTCODE",
    "BUURTNAAM",
    "lane_length_m"
]

existing_lane_join_cols = [
    col for col in desired_lane_join_cols
    if col in lane_intersections.columns
]

lane_neighbourhood_join = lane_intersections[existing_lane_join_cols].copy()

lane_neighbourhood_join = lane_neighbourhood_join.rename(
    columns={"lane_length_m": "intersected_length_m"}
)

lane_neighbourhood_join.to_csv(
    "../data/clean/lane_neighbourhood_join.csv",
    index=False
)

print("Saved lane_neighbourhood_join.csv")
print("Shape:", lane_neighbourhood_join.shape)

lane_neighbourhood_join.head()

Saved lane_neighbourhood_join.csv
Shape: (5467, 12)


,id,element,highway,bicycle,cycleway,cycleway:left,cycleway:right,cycleway:both,source_query,BUURTCODE,BUURTNAAM,intersected_length_m
0,4260673,way,cycleway,NaN,NaN,NaN,NaN,NaN,{'highway': 'cycleway'},115,TU-terrein,36.485973
1,4260684,way,cycleway,NaN,NaN,NaN,NaN,NaN,{'highway': 'cycleway'},115,TU-terrein,7.135659
2,4260685,way,cycleway,NaN,NaN,NaN,NaN,NaN,{'highway': 'cycleway'},336,Karpen,20.118614
3,4260685,way,cycleway,NaN,NaN,NaN,NaN,NaN,{'highway': 'cycleway'},115,TU-terrein,10.100727
4,4260687,way,cycleway,NaN,NaN,NaN,NaN,NaN,{'highway': 'cycleway'},337,Koudenhoven,149.911607


# Aggregate lane length 

In [30]:
lane_summary = (
    lane_neighbourhood_join
    .groupby(["BUURTCODE", "BUURTNAAM"], as_index=False)
    .agg(
        lane_length_m=("intersected_length_m", "sum")
    )
)

lane_summary["has_lane_data"] = True

lane_summary.to_csv(
    "../data/clean/lane_neighbourhood_summary.csv",
    index=False
)

print("Saved lane_neighbourhood_summary.csv")
print("Shape:", lane_summary.shape)

lane_summary.sort_values("lane_length_m", ascending=False).head(10)

Saved lane_neighbourhood_summary.csv
Shape: (111, 4)


,BUURTCODE,BUURTNAAM,lane_length_m,has_lane_data
4,115,TU-terrein,17812.439987,True
69,534,Blixembosch-Oost,13895.592042,True
73,543,Vaartbroek,12597.654222,True
96,637,Flight Forum,11512.189275,True
89,628,Mispelhoef,11435.162989,True
49,431,Generalenbuurt,10503.761855,True
84,622,Het Ven,10238.537857,True
74,544,Heesterakker,10163.873294,True
92,633,Grasrijk,8989.917116,True
62,521,Kerkdorp Acht,8954.094660,True


# Create final neighbourhood summary

In [31]:
neighbourhood_summary = neighbourhoods[
    [
        "BUURTCODE",
        "BUURTNAAM",
        "WIJKNAAM",
        "STADSDEELNAAM",
        "population",
        "area_km2",
        "OAD",
        "pop_density"
    ]
].copy()

neighbourhood_summary = neighbourhood_summary.merge(
    parking_summary,
    on=["BUURTCODE", "BUURTNAAM"],
    how="left"
)

neighbourhood_summary = neighbourhood_summary.merge(
    accident_summary,
    on=["BUURTCODE", "BUURTNAAM"],
    how="left"
)

neighbourhood_summary = neighbourhood_summary.merge(
    lane_summary,
    on=["BUURTCODE", "BUURTNAAM"],
    how="left"
)

neighbourhood_summary.head()

,BUURTCODE,BUURTNAAM,WIJKNAAM,STADSDEELNAAM,population,area_km2,OAD,pop_density,parking_count,parking_capacity,accident_count,injury_count,fatal_count,property_damage_only_count,lane_length_m,has_lane_data
0,522,Achtse Barrier-Gunterslaer,Wijk Achtse Molen,Stadsdeel Woensel-Noord,3773.0,1.015387,3715.825734,3715.825734,NaN,NaN,36.0,9.0,0.0,27.0,8945.836513,True
1,524,Achtse Barrier-Hoeven,Wijk Achtse Molen,Stadsdeel Woensel-Noord,3886.0,0.744209,5221.648554,5221.648554,NaN,NaN,14.0,1.0,0.0,13.0,6662.592343,True
2,523,Achtse Barrier-Spaaihoef,Wijk Achtse Molen,Stadsdeel Woensel-Noord,4524.0,1.015942,4453.010370,4453.010370,NaN,NaN,18.0,4.0,0.0,14.0,6086.678251,True
3,423,Barrier,Wijk Erp,Stadsdeel Woensel-Zuid,2040.0,0.257474,7923.122667,7923.122667,NaN,NaN,30.0,3.0,0.0,27.0,1364.757956,True
4,631,BeA2,Wijk Meerhoven,Stadsdeel Strijp,23.0,2.620186,8.778001,8.778001,NaN,NaN,2.0,1.0,0.0,1.0,7059.812416,True


# Fill missing values

Some neighbourhoods may not have any parking facilities or accident records, so the merged values can be `NaN`. These missing values should be filled with 0.

In [32]:
neighbourhood_summary["parking_count"] = (
    neighbourhood_summary["parking_count"]
    .fillna(0)
    .astype(int)
)

neighbourhood_summary["parking_capacity"] = (
    neighbourhood_summary["parking_capacity"]
    .fillna(0)
)

for col in [
    "accident_count",
    "injury_count",
    "fatal_count",
    "property_damage_only_count"
]:
    neighbourhood_summary[col] = (
        neighbourhood_summary[col]
        .fillna(0)
        .astype(int)
    )

neighbourhood_summary["has_lane_data"] = (
    neighbourhood_summary["has_lane_data"]
    .fillna(False)
    .astype(bool)
)

neighbourhood_summary["lane_length_m"] = (
    neighbourhood_summary["lane_length_m"]
    .fillna(0)
)

neighbourhood_summary.head()

,BUURTCODE,BUURTNAAM,WIJKNAAM,STADSDEELNAAM,population,area_km2,OAD,pop_density,parking_count,parking_capacity,accident_count,injury_count,fatal_count,property_damage_only_count,lane_length_m,has_lane_data
0,522,Achtse Barrier-Gunterslaer,Wijk Achtse Molen,Stadsdeel Woensel-Noord,3773.0,1.015387,3715.825734,3715.825734,0,0.0,36,9,0,27,8945.836513,True
1,524,Achtse Barrier-Hoeven,Wijk Achtse Molen,Stadsdeel Woensel-Noord,3886.0,0.744209,5221.648554,5221.648554,0,0.0,14,1,0,13,6662.592343,True
2,523,Achtse Barrier-Spaaihoef,Wijk Achtse Molen,Stadsdeel Woensel-Noord,4524.0,1.015942,4453.010370,4453.010370,0,0.0,18,4,0,14,6086.678251,True
3,423,Barrier,Wijk Erp,Stadsdeel Woensel-Zuid,2040.0,0.257474,7923.122667,7923.122667,0,0.0,30,3,0,27,1364.757956,True
4,631,BeA2,Wijk Meerhoven,Stadsdeel Strijp,23.0,2.620186,8.778001,8.778001,0,0.0,2,1,0,1,7059.812416,True


# Add indicators

In [36]:
# These indicators are used only as validation/reference values.
# Bike lane length and lane density are not pre-aggregated here,
# because they should be calculated dynamically in Neo4j queries.

import numpy as np

valid_population = (
    neighbourhood_summary["population"].notna()
    & (neighbourhood_summary["population"] > 0)
)

neighbourhood_summary["parking_capacity_per_1000_residents"] = np.nan
neighbourhood_summary["accident_rate_per_1000_residents"] = np.nan

neighbourhood_summary.loc[valid_population, "parking_capacity_per_1000_residents"] = (
    neighbourhood_summary.loc[valid_population, "parking_capacity"]
    / neighbourhood_summary.loc[valid_population, "population"]
    * 1000
)

neighbourhood_summary.loc[valid_population, "accident_rate_per_1000_residents"] = (
    neighbourhood_summary.loc[valid_population, "accident_count"]
    / neighbourhood_summary.loc[valid_population, "population"]
    * 1000
)

neighbourhood_summary.replace([np.inf, -np.inf], np.nan, inplace=True)

neighbourhood_summary.head()

,BUURTCODE,BUURTNAAM,WIJKNAAM,STADSDEELNAAM,population,area_km2,OAD,pop_density,parking_count,parking_capacity,accident_count,injury_count,fatal_count,property_damage_only_count,lane_length_m,has_lane_data,parking_capacity_per_1000_residents,accident_rate_per_1000_residents
0,522,Achtse Barrier-Gunterslaer,Wijk Achtse Molen,Stadsdeel Woensel-Noord,3773.0,1.015387,3715.825734,3715.825734,0,0.0,36,9,0,27,8945.836513,True,0.0,9.541479
1,524,Achtse Barrier-Hoeven,Wijk Achtse Molen,Stadsdeel Woensel-Noord,3886.0,0.744209,5221.648554,5221.648554,0,0.0,14,1,0,13,6662.592343,True,0.0,3.602676
2,523,Achtse Barrier-Spaaihoef,Wijk Achtse Molen,Stadsdeel Woensel-Noord,4524.0,1.015942,4453.010370,4453.010370,0,0.0,18,4,0,14,6086.678251,True,0.0,3.978780
3,423,Barrier,Wijk Erp,Stadsdeel Woensel-Zuid,2040.0,0.257474,7923.122667,7923.122667,0,0.0,30,3,0,27,1364.757956,True,0.0,14.705882
4,631,BeA2,Wijk Meerhoven,Stadsdeel Strijp,23.0,2.620186,8.778001,8.778001,0,0.0,2,1,0,1,7059.812416,True,0.0,86.956522


The three indicators mean:

- `parking_capacity_per_1000_residents`: how much bicycle parking capacity there is per 1,000 residents
- `lane_density_m_per_km2`: how many metres of bike lanes there are per square kilometre
- `accident_rate_per_1000_residents`: how many cycling accidents there are per 1,000 residents

# Validation checks

In [39]:
print("Number of neighbourhoods:", len(neighbourhood_summary))
print("Duplicated BUURTCODE:", neighbourhood_summary["BUURTCODE"].duplicated().sum())

print("Missing parking_count:", neighbourhood_summary["parking_count"].isna().sum())
print("Missing parking_capacity:", neighbourhood_summary["parking_capacity"].isna().sum())
print("Missing accident_count:", neighbourhood_summary["accident_count"].isna().sum())
print("Missing lane_length_m:", neighbourhood_summary["lane_length_m"].isna().sum())

print("Total parking facilities:", neighbourhood_summary["parking_count"].sum())
print("Total parking capacity:", neighbourhood_summary["parking_capacity"].sum())

print("Original accident records:", len(accident_join))
print("Matched accidents:", len(matched_accident_join))
print("Unmatched accidents:", len(unmatched_accidents))
print("Total accidents in summary:", neighbourhood_summary["accident_count"].sum())

print("Total injury accidents:", neighbourhood_summary["injury_count"].sum())
print("Total fatal accidents:", neighbourhood_summary["fatal_count"].sum())
print("Total property-damage-only accidents:", neighbourhood_summary["property_damage_only_count"].sum())

print("Lane-neighbourhood join rows:", len(lane_neighbourhood_join))
print("Neighbourhoods with lane data:", neighbourhood_summary["has_lane_data"].sum())
print("Total lane length from join table:", lane_neighbourhood_join["intersected_length_m"].sum())
print("Total lane length from summary:", lane_summary["lane_length_m"].sum())

print("Lane length totals match:",
      np.isclose(
          lane_neighbourhood_join["intersected_length_m"].sum(),
          lane_summary["lane_length_m"].sum()
      ))

Number of neighbourhoods: 111
Duplicated BUURTCODE: 0
Missing parking_count: 0
Missing parking_capacity: 0
Missing accident_count: 0
Missing lane_length_m: 0
Total parking facilities: 7
Total parking capacity: 7700.0
Original accident records: 1653
Matched accidents: 1621
Unmatched accidents: 32
Total accidents in summary: 1621
Total injury accidents: 341
Total fatal accidents: 4
Total property-damage-only accidents: 1276
Lane-neighbourhood join rows: 5467
Neighbourhoods with lane data: 111
Total lane length from join table: 483302.4786758395
Total lane length from summary: 483302.4786758395
Lane length totals match: True


# Save final output

In [40]:
neighbourhood_summary.to_csv(
    "../data/clean/neighbourhood_summary.csv",
    index=False
)

print("Saved neighbourhood_summary.csv")
print("Shape:", neighbourhood_summary.shape)

Saved neighbourhood_summary.csv
Shape: (111, 18)


The final `gap_score` is not calculated in this notebook. It will be computed in a separate ranking analysis notebook using the integrated `neighbourhood_summary.csv`.

# Summary

This notebook produces the following output files:

**Join tables (segment/record level):**
- `parking_neighbourhood_join.csv`: each parking facility matched to its neighbourhood. Used to create `BikeParkingFacility` nodes and `LOCATED_IN` relationships in Neo4j.
- `accident_neighbourhood_join.csv`: each accident matched to its neighbourhood (1,653 records, including 32 unmatched). Used to create `CyclingAccident` nodes and `OCCURRED_IN` relationships in Neo4j.
- `lane_neighbourhood_join.csv`: each bike lane segment intersected with neighbourhood polygons, with `intersected_length_m` per segment. This is the primary input for Neo4j, where lane length per neighbourhood is calculated dynamically at query time.

**Summary tables (neighbourhood level, for validation):**
- `parking_neighbourhood_summary.csv`: `parking_count` and `parking_capacity` per neighbourhood.
- `accident_neighbourhood_summary.csv`: `accident_count`, `injury_count`, `fatal_count`, and `property_damage_only_count` per neighbourhood.
- `lane_neighbourhood_summary.csv`: `lane_length_m` per neighbourhood. Used as a validation reference to cross-check Neo4j query results.

**Final integrated table:**
- `neighbourhood_summary.csv`: one row per neighbourhood, merging demographic attributes with parking, accident, and lane indicators. Includes derived reference metrics (`parking_capacity_per_1000_residents`, `accident_rate_per_1000_residents`). Lane density (`lane_density_m_per_km2`) is intentionally not pre-computed here; it should be calculated in Neo4j or in the ranking notebook.

**Design decision:** Bike lane aggregation is kept at the segment level in the join table so that Neo4j queries can compute lane length dynamically, with the option to filter by lane type, highway category, or other attributes. The pre-aggregated `lane_neighbourhood_summary.csv` and the lane columns in `neighbourhood_summary.csv` serve only as validation benchmarks to verify that Neo4j query results are consistent with the Python pipeline output.

The final `gap_score` is not calculated in this notebook. It will be computed in a separate ranking analysis using the knowledge graph or the integrated `neighbourhood_summary.csv`.